In [1]:
import os
import json
import copy
import math
import random
import warnings
import hashlib
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List, Optional, Tuple

from joblib import dump, load

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch_geometric.explain import Explainer, GNNExplainer  
from gprofiler import GProfiler 

import shap  

import gseapy as gp  

from captum.attr import IntegratedGradients 

from IPython.display import display

from sklearn.decomposition import PCA as CPU_PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesRegressor

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 200)

BASE_SEED = 19537
ALL_SEEDS = [19537, 1584678, 17052356]

SEED = BASE_SEED
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

RNG = np.random.default_rng(SEED)


def set_seeds(seed: int) -> None:
    global SEED, RNG
    SEED = int(seed)
    random.seed(SEED)
    np.random.seed(SEED)
    os.environ["PYTHONHASHSEED"] = str(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    RNG = np.random.default_rng(SEED)


warnings.filterwarnings("ignore", category=UserWarning)
TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Torch device: {TORCH_DEVICE}")

# Paths 
ARTIFACTS = Path("artifacts")
IN_CLEAN = ARTIFACTS / "cleaned" / "notebook 2"
IN_T2 = ARTIFACTS / "aligned" / "notebook 1" / "track2_nonintersection"
IN_META = ARTIFACTS / "metadata"

NOTEBOOK_SUBDIR = "notebook 7_notmiwae_prot"
OUT_REPORTS = ARTIFACTS / "reports" / NOTEBOOK_SUBDIR
OUT_META = ARTIFACTS / "metadata" / NOTEBOOK_SUBDIR
OUT_CACHE = ARTIFACTS / "cache" / NOTEBOOK_SUBDIR

for d in [OUT_REPORTS, OUT_META, OUT_CACHE]:
    d.mkdir(parents=True, exist_ok=True)

NOTMIWAE_LOCK_FILENAME = "notmiwae_imputer_choice_prot.json"

# Backbone 
LOCK_PATH = IN_META / "proteomics_backbone_lock.json"
if not LOCK_PATH.exists():
    raise FileNotFoundError(
        f"proteomics_backbone_lock.json not found at {LOCK_PATH}. "
        "Run the lock cell at the end of Notebook 3b first."
    )

with LOCK_PATH.open("r", encoding="utf-8") as f:
    backbone_lock = json.load(f)

PRIMARY_ARM = backbone_lock["track1_primary_arm"]
SECONDARY_ARM = backbone_lock["track1_secondary_arm"]
TRACK2_ARM = backbone_lock["track2_stress_test_arm"]
DEPRIORITISED_ARM = backbone_lock.get("deprioritised_arm", "prot_rppa_ccle")

print("Backbone lock loaded.")
print(f"  Primary arm  : {PRIMARY_ARM}")
print(f"  Secondary arm: {SECONDARY_ARM}")
print(f"  Track 2 arm  : {TRACK2_ARM}")
print(f"  Seeds to run : {ALL_SEEDS}")

# Sparse-MS-oriented default: primary sparse arm + secondary sparse arm + robustness arm + deprioritised arm
ACTIVE_ARMS = sorted(list({PRIMARY_ARM, SECONDARY_ARM, TRACK2_ARM, DEPRIORITISED_ARM}))

FULL_FEATURE_SETS = [
    "rna",
    "cnv",
    "mut",
    "prot",
    "rna+cnv",
    "rna+mut",
    "rna+prot",
    "cnv+mut",
    "cnv+prot",
    "mut+prot",
    "rna+cnv+mut",
    "rna+cnv+prot",
    "rna+mut+prot",
    "cnv+mut+prot",
    "rna+cnv+mut+prot",
]

ACTIVE_MODELS = ["elasticnet", "ridge", "extratrees"]

NOTMIWAE_TEST_CONFIGS = [
    {"rank": rank, "arm": arm, "model": model, "feature_set": fs}
    for rank, (arm, model, fs) in enumerate(
        [
            (arm, model, fs)
            for arm in ACTIVE_ARMS
            for model in ACTIVE_MODELS
            for fs in FULL_FEATURE_SETS
        ],
        start=1,
    )
]

CONFIGS_BY_ARM = {
    arm: [cfg for cfg in NOTMIWAE_TEST_CONFIGS if cfg["arm"] == arm]
    for arm in ACTIVE_ARMS
}

print("\nnot-MIWAE configuration grid loaded.")
print("Active arms:", ACTIVE_ARMS)
print("Active models:", ACTIVE_MODELS)
print("Feature sets:", FULL_FEATURE_SETS)
display(pd.DataFrame(NOTMIWAE_TEST_CONFIGS))

Torch device: cuda
Backbone lock loaded.
  Primary arm  : prot_procan_depmapSanger
  Secondary arm: prot_ms_ccle_gygi
  Track 2 arm  : prot_combined_union
  Seeds to run : [19537, 1584678, 17052356]

not-MIWAE configuration grid loaded.
Active arms: ['prot_combined_union', 'prot_ms_ccle_gygi', 'prot_procan_depmapSanger', 'prot_rppa_ccle']
Active models: ['elasticnet', 'ridge', 'extratrees']
Feature sets: ['rna', 'cnv', 'mut', 'prot', 'rna+cnv', 'rna+mut', 'rna+prot', 'cnv+mut', 'cnv+prot', 'mut+prot', 'rna+cnv+mut', 'rna+cnv+prot', 'rna+mut+prot', 'cnv+mut+prot', 'rna+cnv+mut+prot']


,rank,arm,model,feature_set
0,1,prot_combined_union,elasticnet,rna
1,2,prot_combined_union,elasticnet,cnv
2,3,prot_combined_union,elasticnet,mut
3,4,prot_combined_union,elasticnet,prot
4,5,prot_combined_union,elasticnet,rna+cnv
...,...,...,...,...
175,176,prot_rppa_ccle,extratrees,rna+cnv+mut
176,177,prot_rppa_ccle,extratrees,rna+cnv+prot
177,178,prot_rppa_ccle,extratrees,rna+mut+prot
178,179,prot_rppa_ccle,extratrees,cnv+mut+prot


In [2]:
# Settings
N_DRUGS_BAKEOFF = 100
MIN_CELLS_PER_DRUG = 80
N_SPLITS_DESIRED = 10
PRIMARY_TARGET = "auc"
PCA_COMPONENTS = 100

RIDGE_ALPHA = 1.0
EN_ALPHA = 0.05
EN_L1_RATIO = 0.2
ET_N_ESTIMATORS = 400
ET_MAX_DEPTH = None
ET_MIN_SAMPLES_LEAF = 2

# not-MIWAE settings
NOTMIWAE_LATENT_DIM = 48
NOTMIWAE_HIDDEN_DIM = 256
NOTMIWAE_DROPOUT = 0.10
NOTMIWAE_EPOCHS = 180
NOTMIWAE_PATIENCE = 20
NOTMIWAE_BATCH_SIZE = 64
NOTMIWAE_LR = 1e-3
NOTMIWAE_WEIGHT_DECAY = 1e-5
NOTMIWAE_TRAIN_K = 20
NOTMIWAE_IMPUTE_K = 64
NOTMIWAE_IMPUTE_BATCH_SIZE = 64
NOTMIWAE_MIN_LOGVAR = -8.0
NOTMIWAE_MAX_LOGVAR = 3.0
VAL_FRAC = 0.15
MIN_VAL_SAMPLES = 24

USE_AUX_FOR_PROT_IMPUTATION = True
AUX_FOR_PROT_NAME = "aux_rna_cnv_mut"

NOTMIWAE_STRATEGIES = [
    {"name": "notmiwae_selfmask", "add_indicators": False, "missing_model": "selfmask"},
    {"name": "notmiwae_selfmask+indicators", "add_indicators": True, "missing_model": "selfmask"},
    {"name": "notmiwae_linear", "add_indicators": False, "missing_model": "linear"},
    {"name": "notmiwae_linear+indicators", "add_indicators": True, "missing_model": "linear"},
]

CACHE_VERSION = "v1_notmiwae_prot"
CACHE_TAG = (
    f"{CACHE_VERSION}"
    f"__target{PRIMARY_TARGET}"
    f"__splits{N_SPLITS_DESIRED}"
    f"__ndrugs{N_DRUGS_BAKEOFF}"
    f"__mindrug{MIN_CELLS_PER_DRUG}"
    f"__pca{PCA_COMPONENTS}"
    f"__lat{NOTMIWAE_LATENT_DIM}"
    f"__hid{NOTMIWAE_HIDDEN_DIM}"
    f"__k{NOTMIWAE_TRAIN_K}"
    f"__ep{NOTMIWAE_EPOCHS}"
    f"__bs{NOTMIWAE_BATCH_SIZE}"
    f"__lr{NOTMIWAE_LR}"
)

In [3]:
# Helpers
def read_parquet_strict(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing parquet: {path}")
    return pd.read_parquet(path)


def normalise_str_index(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.index = df.index.astype(str)
    return df


def write_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def spearman_corr(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if y_true.size < 2:
        return np.nan
    rt = pd.Series(y_true).rank(method="average").to_numpy(dtype=float)
    rp = pd.Series(y_pred).rank(method="average").to_numpy(dtype=float)
    if np.std(rt) == 0 or np.std(rp) == 0:
        return np.nan
    return float(np.corrcoef(rt, rp)[0, 1])


def safe_nanmean(x) -> float:
    x = pd.Series(x).to_numpy(dtype=float)
    x = x[np.isfinite(x)]
    return float(x.mean()) if x.size > 0 else np.nan


def safe_nanmedian(x) -> float:
    x = pd.Series(x).to_numpy(dtype=float)
    x = x[np.isfinite(x)]
    return float(np.median(x)) if x.size > 0 else np.nan


def safe_nanstd(x) -> float:
    x = pd.Series(x).to_numpy(dtype=float)
    x = x[np.isfinite(x)]
    return float(x.std()) if x.size > 0 else np.nan


def pick_group_column(cell_index: pd.DataFrame) -> str:
    for c in ["lineage_1", "primary_disease", "lineage", "lineage_2"]:
        if c in cell_index.columns:
            return c
    return "depmap_id"


def safe_group_splits(
    cells: List[str],
    groups: pd.Series,
    n_splits_desired: int,
    seed: int,
) -> Tuple[List[Tuple[np.ndarray, np.ndarray]], str]:
    groups = groups.reindex(cells).fillna("Unknown").astype(str)
    n_groups = groups.nunique()
    n_cells = len(cells)
    n_splits = min(n_splits_desired, n_groups, n_cells)
    if n_splits >= 2 and n_groups >= 2:
        sp = GroupKFold(n_splits=n_splits)
        return (
            list(sp.split(np.zeros((n_cells, 1)), np.zeros(n_cells), groups.values)),
            f"GroupKFold(n_splits={n_splits})",
        )
    n_splits = min(max(2, n_splits), n_cells)
    sp = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(sp.split(np.zeros((n_cells, 1)))), f"KFold(n_splits={n_splits})"


def parse_feature_set(feature_set: str) -> Tuple[str, ...]:
    if feature_set is None:
        return tuple()
    feature_set = str(feature_set).strip()
    if feature_set == "":
        return tuple()
    return tuple(feature_set.split("+"))


def concat_selected_modalities(
    mats: Dict[str, np.ndarray],
    selected_keys: Tuple[str, ...],
    n_rows: int,
) -> np.ndarray:
    parts = []
    for k in selected_keys:
        arr = mats.get(k)
        if arr is None or arr.shape[1] == 0:
            return np.zeros((n_rows, 0), dtype=np.float32)
        parts.append(arr)
    if not parts:
        return np.zeros((n_rows, 0), dtype=np.float32)
    return np.concatenate(parts, axis=1)


def make_model(model_name: str, seed: int):
    model_name = str(model_name).lower()

    if model_name == "ridge":
        return Ridge(alpha=RIDGE_ALPHA, solver="svd")

    if model_name == "elasticnet":
        return ElasticNet(
            alpha=EN_ALPHA,
            l1_ratio=EN_L1_RATIO,
            random_state=seed,
            max_iter=10000,
        )

    if model_name == "extratrees":
        return ExtraTreesRegressor(
            n_estimators=ET_N_ESTIMATORS,
            random_state=seed,
            n_jobs=-1,
            max_depth=ET_MAX_DEPTH,
            min_samples_leaf=ET_MIN_SAMPLES_LEAF,
        )

    raise ValueError(f"Unsupported model for Notebook 7 not-MIWAE bake-off: {model_name}")


def _safe_name(s: str) -> str:
    s = str(s)
    return (
        s.replace(os.sep, "_")
         .replace(" ", "_")
         .replace("+", "plus")
         .replace(":", "_")
         .replace("/", "_")
         .replace("\\", "_")
    )


def _cache_digest(**parts) -> str:
    payload = json.dumps(parts, sort_keys=True, default=str, separators=(",", ":"))
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()[:20]


def _cache_file(dirpath: Path, prefix: str, **parts) -> Path:
    dirpath.mkdir(parents=True, exist_ok=True)
    digest = _cache_digest(**parts)
    return dirpath / f"{prefix}__{digest}.joblib"


def _eval_cache_dir(kind: str) -> Path:
    path = OUT_CACHE / kind
    path.mkdir(parents=True, exist_ok=True)
    return path


def _eval_cache_path(
    kind: str,
    seed: int,
    arm: str,
    drug: str,
    fold_i: int,
    cfg_rank: int,
    model_name: str,
    feature_set: str,
    imputer_strategy: str,
) -> Path:
    return _cache_file(
        _eval_cache_dir(kind),
        "eval",
        cache_tag=CACHE_TAG,
        kind=kind,
        seed=seed,
        arm=arm,
        drug=drug,
        fold_i=fold_i,
        cfg_rank=cfg_rank,
        model_name=model_name,
        feature_set=feature_set,
        imputer_strategy=imputer_strategy,
    )


def load_or_run_eval_cache(
    *,
    kind: str,
    seed: int,
    arm: str,
    drug: str,
    fold_i: int,
    cfg_rank: int,
    model_name: str,
    feature_set: str,
    imputer_strategy: str,
    extra_meta: dict,
    X_train: np.ndarray,
    X_test: np.ndarray,
    y_train: np.ndarray,
    y_test: np.ndarray,
) -> dict:
    path = _eval_cache_path(
        kind=kind,
        seed=seed,
        arm=arm,
        drug=drug,
        fold_i=fold_i,
        cfg_rank=cfg_rank,
        model_name=model_name,
        feature_set=feature_set,
        imputer_strategy=imputer_strategy,
    )
    if path.exists():
        return load(path)

    mdl = make_model(model_name, seed)
    mdl.fit(X_train, y_train)
    pred = mdl.predict(X_test)

    row = {
        "seed": seed,
        "config_rank": cfg_rank,
        "arm": arm,
        "model": model_name,
        "feature_set": feature_set,
        "compound_id": drug,
        "fold": fold_i,
        "imputer_strategy": imputer_strategy,
        "n_train": int(len(y_train)),
        "n_test": int(len(y_test)),
        "spearman": spearman_corr(y_test, pred),
        "r2": float(r2_score(y_test, pred)),
        **extra_meta,
    }
    dump(row, path)
    return row

In [4]:
# Fold-safe modality transforms for RNA/CNV/MUT
class BaseModalityTransformer:
    def __init__(self, n_components: int, random_state: int):
        self.n_components = n_components
        self.random_state = random_state
        self._imp = SimpleImputer(strategy="median")
        self._scaler = StandardScaler()
        self._pca = None
        self._keep = None

    def fit(self, X: np.ndarray) -> "BaseModalityTransformer":
        X = np.asarray(X, dtype=float)
        self._keep = np.isfinite(X).any(axis=0)
        if self._keep.sum() == 0:
            return self
        Xk = X[:, self._keep]
        Xs = self._scaler.fit_transform(self._imp.fit_transform(Xk))
        n, d = Xs.shape
        n_comp = min(self.n_components, max(1, n - 1), d)
        self._pca = CPU_PCA(n_components=n_comp, random_state=self.random_state)
        self._pca.fit(Xs)
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=float)
        if self._keep is None or self._keep.sum() == 0:
            return np.zeros((X.shape[0], 0), dtype=np.float32)
        Xk = X[:, self._keep]
        Xs = self._scaler.transform(self._imp.transform(Xk))
        if self._pca is not None:
            Xs = self._pca.transform(Xs)
        return Xs.astype(np.float32)


class StandardiseObservedBlock:
    def __init__(self):
        self.mean_ = None
        self.std_ = None
        self.keep_ = None

    def fit(self, X: np.ndarray) -> "StandardiseObservedBlock":
        X = np.asarray(X, dtype=float)
        self.keep_ = np.isfinite(X).any(axis=0)
        if self.keep_ is None or self.keep_.sum() == 0:
            self.mean_ = np.zeros(0, dtype=np.float32)
            self.std_ = np.ones(0, dtype=np.float32)
            return self
        Xk = X[:, self.keep_]
        col_mean = np.nanmean(Xk, axis=0)
        col_std = np.nanstd(Xk, axis=0)
        col_std = np.where(col_std <= 1e-8, 1.0, col_std)
        self.mean_ = col_mean.astype(np.float32)
        self.std_ = col_std.astype(np.float32)
        return self

    def transform(self, X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        X = np.asarray(X, dtype=float)
        if self.keep_ is None or self.keep_.sum() == 0:
            return (
                np.zeros((X.shape[0], 0), dtype=np.float32),
                np.zeros((X.shape[0], 0), dtype=bool),
            )
        Xk = X[:, self.keep_]
        miss = ~np.isfinite(Xk)
        Xfill = np.where(miss, self.mean_[None, :], Xk)
        Xstd = (Xfill - self.mean_[None, :]) / self.std_[None, :]
        return Xstd.astype(np.float32), miss

    def inverse_transform(self, X_std: np.ndarray) -> np.ndarray:
        X_std = np.asarray(X_std, dtype=np.float32)
        if X_std.shape[1] == 0:
            return X_std.astype(np.float32)
        return (X_std * self.std_[None, :] + self.mean_[None, :]).astype(np.float32)

In [5]:
# not-MIWAE model
class GaussianNotMIWAE(nn.Module):
    def __init__(
        self,
        input_dim: int,
        latent_dim: int,
        hidden_dim: int,
        aux_dim: int = 0,
        missing_model: str = "selfmask",
        dropout: float = 0.10,
        min_logvar: float = -8.0,
        max_logvar: float = 3.0,
    ):
        super().__init__()
        self.input_dim = int(input_dim)
        self.latent_dim = int(latent_dim)
        self.hidden_dim = int(hidden_dim)
        self.aux_dim = int(aux_dim)
        self.missing_model = str(missing_model)
        self.min_logvar = float(min_logvar)
        self.max_logvar = float(max_logvar)

        enc_in = self.input_dim * 2 + self.aux_dim
        self.encoder = nn.Sequential(
            nn.Linear(enc_in, self.hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(self.hidden_dim, self.hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.q_mu = nn.Linear(self.hidden_dim, self.latent_dim)
        self.q_logvar = nn.Linear(self.hidden_dim, self.latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(self.latent_dim + self.aux_dim, self.hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(self.hidden_dim, self.hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.p_mu = nn.Linear(self.hidden_dim, self.input_dim)
        self.p_logvar = nn.Linear(self.hidden_dim, self.input_dim)

        if self.missing_model == "selfmask":
            self.miss_w = nn.Parameter(torch.zeros(self.input_dim))
            self.miss_b = nn.Parameter(torch.zeros(self.input_dim))
        elif self.missing_model == "linear":
            self.miss_linear = nn.Linear(self.input_dim + self.aux_dim, self.input_dim)
        elif self.missing_model == "nonlinear":
            miss_in = self.input_dim + self.aux_dim
            self.miss_nonlinear = nn.Sequential(
                nn.Linear(miss_in, self.hidden_dim),
                nn.Tanh(),
                nn.Dropout(dropout),
                nn.Linear(self.hidden_dim, self.input_dim),
            )
        else:
            raise ValueError(f"Unsupported missing model: {self.missing_model}")

    def encode(
        self,
        x_filled: torch.Tensor,
        obs_mask: torch.Tensor,
        aux: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        parts = [x_filled, obs_mask]
        if aux is not None and aux.numel() > 0:
            parts.append(aux)
        h = self.encoder(torch.cat(parts, dim=-1))
        mu = self.q_mu(h)
        logvar = torch.clamp(self.q_logvar(h), min=-10.0, max=10.0)
        return mu, logvar

    def sample_z(self, mu: torch.Tensor, logvar: torch.Tensor, k: int) -> torch.Tensor:
        std = torch.exp(0.5 * logvar)
        eps = torch.randn(mu.shape[0], k, mu.shape[1], device=mu.device)
        return mu.unsqueeze(1) + eps * std.unsqueeze(1)

    def decode(
        self,
        z: torch.Tensor,
        aux: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        if aux is not None and aux.numel() > 0:
            aux_exp = aux.unsqueeze(1).expand(z.shape[0], z.shape[1], aux.shape[1])
            z_in = torch.cat([z, aux_exp], dim=-1)
        else:
            z_in = z
        h = self.decoder(z_in)
        mu_x = self.p_mu(h)
        logvar_x = torch.clamp(self.p_logvar(h), min=self.min_logvar, max=self.max_logvar)
        return mu_x, logvar_x

    def missing_logits(
        self,
        x_completed: torch.Tensor,
        aux: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        if self.missing_model == "selfmask":
            return x_completed * self.miss_w.view(1, 1, -1) + self.miss_b.view(1, 1, -1)

        if aux is not None and aux.numel() > 0:
            aux_exp = aux.unsqueeze(1).expand(x_completed.shape[0], x_completed.shape[1], aux.shape[1])
            x_in = torch.cat([x_completed, aux_exp], dim=-1)
        else:
            x_in = x_completed

        if self.missing_model == "linear":
            return self.miss_linear(x_in)
        return self.miss_nonlinear(x_in)

    def bound(
        self,
        x_obs: torch.Tensor,
        obs_mask: torch.Tensor,
        aux: Optional[torch.Tensor] = None,
        k: int = 20,
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        x_filled = x_obs * obs_mask
        mu_z, logvar_z = self.encode(x_filled=x_filled, obs_mask=obs_mask, aux=aux)
        z = self.sample_z(mu_z, logvar_z, k=k)
        mu_x, logvar_x = self.decode(z=z, aux=aux)

        std_x = torch.exp(0.5 * logvar_x)
        eps_x = torch.randn_like(mu_x)
        x_sample = mu_x + eps_x * std_x

        x_obs_exp = x_obs.unsqueeze(1)
        obs_exp = obs_mask.unsqueeze(1)
        x_completed = obs_exp * x_obs_exp + (1.0 - obs_exp) * x_sample

        log_p_x = -0.5 * (
            math.log(2.0 * math.pi) + logvar_x + (x_obs_exp - mu_x) ** 2 / torch.exp(logvar_x)
        )
        log_p_x_obs = (log_p_x * obs_exp).sum(dim=-1)

        miss_logits = self.missing_logits(x_completed=x_completed, aux=aux)
        log_p_s = -F.binary_cross_entropy_with_logits(
            miss_logits,
            obs_exp.expand_as(miss_logits),
            reduction="none",
        ).sum(dim=-1)

        log_p_z = -0.5 * (math.log(2.0 * math.pi) + z.pow(2)).sum(dim=-1)
        log_q_z = -0.5 * (
            math.log(2.0 * math.pi)
            + logvar_z.unsqueeze(1)
            + (z - mu_z.unsqueeze(1)) ** 2 / torch.exp(logvar_z.unsqueeze(1))
        ).sum(dim=-1)

        log_w = log_p_x_obs + log_p_s + log_p_z - log_q_z
        bound = torch.logsumexp(log_w, dim=1) - math.log(float(k))
        stats = {
            "bound": float(bound.mean().detach().cpu().item()),
            "log_p_x_obs": float(log_p_x_obs.mean().detach().cpu().item()),
            "log_p_s": float(log_p_s.mean().detach().cpu().item()),
            "log_p_z": float(log_p_z.mean().detach().cpu().item()),
            "log_q_z": float(log_q_z.mean().detach().cpu().item()),
        }
        return bound.mean(), stats

    @torch.no_grad()
    def posterior_mean_impute(
        self,
        x_obs: torch.Tensor,
        obs_mask: torch.Tensor,
        aux: Optional[torch.Tensor] = None,
        k: int = 64,
    ) -> np.ndarray:
        self.eval()
        x_filled = x_obs * obs_mask
        mu_z, logvar_z = self.encode(x_filled=x_filled, obs_mask=obs_mask, aux=aux)
        z = self.sample_z(mu_z, logvar_z, k=k)
        mu_x, logvar_x = self.decode(z=z, aux=aux)
        std_x = torch.exp(0.5 * logvar_x)
        eps_x = torch.randn_like(mu_x)
        x_sample = mu_x + eps_x * std_x

        x_obs_exp = x_obs.unsqueeze(1)
        obs_exp = obs_mask.unsqueeze(1)
        x_completed = obs_exp * x_obs_exp + (1.0 - obs_exp) * x_sample

        log_p_x = -0.5 * (
            math.log(2.0 * math.pi) + logvar_x + (x_obs_exp - mu_x) ** 2 / torch.exp(logvar_x)
        )
        log_p_x_obs = (log_p_x * obs_exp).sum(dim=-1)
        miss_logits = self.missing_logits(x_completed=x_completed, aux=aux)
        log_p_s = -F.binary_cross_entropy_with_logits(
            miss_logits,
            obs_exp.expand_as(miss_logits),
            reduction="none",
        ).sum(dim=-1)
        log_p_z = -0.5 * (math.log(2.0 * math.pi) + z.pow(2)).sum(dim=-1)
        log_q_z = -0.5 * (
            math.log(2.0 * math.pi)
            + logvar_z.unsqueeze(1)
            + (z - mu_z.unsqueeze(1)) ** 2 / torch.exp(logvar_z.unsqueeze(1))
        ).sum(dim=-1)

        log_w = log_p_x_obs + log_p_s + log_p_z - log_q_z
        w = torch.softmax(log_w, dim=1)
        x_post_mean = (w.unsqueeze(-1) * mu_x).sum(dim=1)
        x_imp = x_obs.clone()
        missing = obs_mask < 0.5
        x_imp[missing] = x_post_mean[missing]
        return x_imp.detach().cpu().numpy().astype(np.float32)


class NotMIWAEImputer:
    def __init__(
        self,
        latent_dim: int,
        hidden_dim: int,
        missing_model: str,
        dropout: float,
        train_k: int,
        impute_k: int,
        lr: float,
        weight_decay: float,
        epochs: int,
        patience: int,
        batch_size: int,
        min_logvar: float,
        max_logvar: float,
        device: torch.device,
    ):
        self.latent_dim = int(latent_dim)
        self.hidden_dim = int(hidden_dim)
        self.missing_model = str(missing_model)
        self.dropout = float(dropout)
        self.train_k = int(train_k)
        self.impute_k = int(impute_k)
        self.lr = float(lr)
        self.weight_decay = float(weight_decay)
        self.epochs = int(epochs)
        self.patience = int(patience)
        self.batch_size = int(batch_size)
        self.min_logvar = float(min_logvar)
        self.max_logvar = float(max_logvar)
        self.device = device
        self.model = None

    def _split_train_val_idx(self, n_samples: int, seed: int) -> Tuple[np.ndarray, np.ndarray]:
        idx = np.arange(n_samples)
        if n_samples < (MIN_VAL_SAMPLES * 2):
            return idx, np.array([], dtype=int)
        rng = np.random.default_rng(seed)
        rng.shuffle(idx)
        n_val = max(MIN_VAL_SAMPLES, int(round(n_samples * VAL_FRAC)))
        n_val = min(n_val, n_samples // 3)
        val_idx = np.sort(idx[:n_val])
        tr_idx = np.sort(idx[n_val:])
        return tr_idx, val_idx

    def fit(
        self,
        X_train_std: np.ndarray,
        miss_train: np.ndarray,
        seed: int,
        aux_train: Optional[np.ndarray] = None,
    ) -> "NotMIWAEImputer":
        input_dim = int(X_train_std.shape[1])
        aux_dim = 0 if aux_train is None else int(aux_train.shape[1])
        self.model = GaussianNotMIWAE(
            input_dim=input_dim,
            latent_dim=self.latent_dim,
            hidden_dim=self.hidden_dim,
            aux_dim=aux_dim,
            missing_model=self.missing_model,
            dropout=self.dropout,
            min_logvar=self.min_logvar,
            max_logvar=self.max_logvar,
        ).to(self.device)

        tr_idx, val_idx = self._split_train_val_idx(X_train_std.shape[0], seed)

        x_tr = torch.from_numpy(X_train_std[tr_idx].astype(np.float32))
        m_tr = torch.from_numpy((~miss_train[tr_idx]).astype(np.float32))
        if aux_train is not None:
            a_tr = torch.from_numpy(aux_train[tr_idx].astype(np.float32))
            train_ds = TensorDataset(x_tr, m_tr, a_tr)
        else:
            train_ds = TensorDataset(x_tr, m_tr)

        train_dl = DataLoader(
            train_ds,
            batch_size=min(self.batch_size, len(train_ds)),
            shuffle=True,
            drop_last=False,
        )

        if len(val_idx) > 0:
            x_val = torch.from_numpy(X_train_std[val_idx].astype(np.float32)).to(self.device)
            m_val = torch.from_numpy((~miss_train[val_idx]).astype(np.float32)).to(self.device)
            a_val = None if aux_train is None else torch.from_numpy(aux_train[val_idx].astype(np.float32)).to(self.device)
        else:
            x_val = None
            m_val = None
            a_val = None

        opt = torch.optim.Adam(self.model.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        best_state = copy.deepcopy(self.model.state_dict())
        best_loss = math.inf
        bad_epochs = 0

        for _epoch in range(self.epochs):
            self.model.train()
            epoch_losses = []
            for batch in train_dl:
                if aux_train is not None:
                    xb, mb, ab = batch
                    ab = ab.to(self.device)
                else:
                    xb, mb = batch
                    ab = None
                xb = xb.to(self.device)
                mb = mb.to(self.device)

                bound, _ = self.model.bound(x_obs=xb, obs_mask=mb, aux=ab, k=self.train_k)
                loss = -bound
                opt.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=5.0)
                opt.step()
                epoch_losses.append(float(loss.detach().cpu().item()))

            self.model.eval()
            with torch.no_grad():
                if x_val is not None:
                    val_bound, _ = self.model.bound(x_obs=x_val, obs_mask=m_val, aux=a_val, k=self.train_k)
                    current = float(-val_bound.detach().cpu().item())
                else:
                    current = float(np.mean(epoch_losses)) if epoch_losses else math.inf

            if current < (best_loss - 1e-6):
                best_loss = current
                best_state = copy.deepcopy(self.model.state_dict())
                bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= self.patience:
                    break

        self.model.load_state_dict(best_state)
        self.model.eval()
        return self

    def impute(
        self,
        X_std: np.ndarray,
        miss: np.ndarray,
        aux: Optional[np.ndarray] = None,
        batch_size: Optional[int] = None,
    ) -> np.ndarray:
        if self.model is None:
            raise RuntimeError("Call fit before impute.")

        X_std = np.asarray(X_std, dtype=np.float32)
        miss = np.asarray(miss, dtype=bool)

        if X_std.shape[0] == 0:
            return np.zeros_like(X_std, dtype=np.float32)

        bs = int(batch_size or NOTMIWAE_IMPUTE_BATCH_SIZE)
        bs = max(1, bs)

        outs = []
        self.model.eval()

        for start in range(0, X_std.shape[0], bs):
            end = min(start + bs, X_std.shape[0])

            x_t = torch.from_numpy(X_std[start:end]).to(self.device, non_blocking=True)
            m_t = torch.from_numpy((~miss[start:end]).astype(np.float32)).to(self.device, non_blocking=True)

            if aux is None:
                a_t = None
            else:
                a_t = torch.from_numpy(
                    np.asarray(aux[start:end], dtype=np.float32)
                ).to(self.device, non_blocking=True)

            out = self.model.posterior_mean_impute(
                x_obs=x_t,
                obs_mask=m_t,
                aux=a_t,
                k=self.impute_k,
            )
            outs.append(out)

            del x_t, m_t, a_t
            if self.device.type == "cuda":
                torch.cuda.empty_cache()

        return np.concatenate(outs, axis=0).astype(np.float32, copy=False)

In [6]:
# not-MIWAE block wrapper and caches
def _base_cache_path(seed: int, arm: str, fold_i: int) -> Path:
    return _cache_file(
        OUT_CACHE,
        "base",
        cache_tag=CACHE_TAG,
        seed=seed,
        arm=arm,
        fold_i=fold_i,
    )


def load_or_build_base_fold_cache(
    seed: int,
    arm: str,
    fold_i: int,
    train_cells: List[str],
    eligible_cells: List[str],
    rna_df: pd.DataFrame,
    cnv_df: pd.DataFrame,
    mut_df: pd.DataFrame,
) -> dict:
    path = _base_cache_path(seed, arm, fold_i)
    if path.exists():
        return load(path)

    rna_tr = BaseModalityTransformer(PCA_COMPONENTS, seed + 0).fit(
        rna_df.loc[train_cells].to_numpy(dtype=float)
    )
    cnv_tr = BaseModalityTransformer(PCA_COMPONENTS, seed + 1).fit(
        cnv_df.loc[train_cells].to_numpy(dtype=float)
    )
    mut_tr = BaseModalityTransformer(PCA_COMPONENTS, seed + 2).fit(
        mut_df.loc[train_cells].to_numpy(dtype=float)
    )

    payload = {
        "Xr": rna_tr.transform(rna_df.loc[eligible_cells].to_numpy(dtype=float)),
        "Xc": cnv_tr.transform(cnv_df.loc[eligible_cells].to_numpy(dtype=float)),
        "Xm": mut_tr.transform(mut_df.loc[eligible_cells].to_numpy(dtype=float)),
    }
    dump(payload, path)
    return payload


def notmiwae_impute_block(
    X_train: np.ndarray,
    X_all: np.ndarray,
    random_state: int,
    missing_model: str,
    add_indicators: bool,
    force_indicators: bool,
    n_pca_components: int,
    aux_train: Optional[np.ndarray] = None,
    aux_all: Optional[np.ndarray] = None,
) -> np.ndarray:
    scaler = StandardiseObservedBlock().fit(X_train)
    Xtr_std, miss_tr = scaler.transform(X_train)
    Xal_std, miss_al = scaler.transform(X_all)

    if Xtr_std.shape[1] == 0:
        return np.zeros((X_all.shape[0], 0), dtype=np.float32)

    model = NotMIWAEImputer(
        latent_dim=NOTMIWAE_LATENT_DIM,
        hidden_dim=NOTMIWAE_HIDDEN_DIM,
        missing_model=missing_model,
        dropout=NOTMIWAE_DROPOUT,
        train_k=NOTMIWAE_TRAIN_K,
        impute_k=NOTMIWAE_IMPUTE_K,
        lr=NOTMIWAE_LR,
        weight_decay=NOTMIWAE_WEIGHT_DECAY,
        epochs=NOTMIWAE_EPOCHS,
        patience=NOTMIWAE_PATIENCE,
        batch_size=NOTMIWAE_BATCH_SIZE,
        min_logvar=NOTMIWAE_MIN_LOGVAR,
        max_logvar=NOTMIWAE_MAX_LOGVAR,
        device=TORCH_DEVICE,
    ).fit(
        X_train_std=Xtr_std,
        miss_train=miss_tr,
        seed=random_state,
        aux_train=aux_train,
    )

    Xtr_imp = model.impute(Xtr_std, miss_tr, aux=aux_train)
    Xal_imp = model.impute(Xal_std, miss_al, aux=aux_all)

    if add_indicators or force_indicators:
        ind_tr = miss_tr.astype(np.float32)
        ind_al = miss_al.astype(np.float32)
        ind_any = ind_tr.any(axis=0)
        ind_tr = ind_tr[:, ind_any]
        ind_al = ind_al[:, ind_any]
        if ind_tr.shape[1] > 0:
            ind_sc = StandardScaler()
            Xtr_imp = np.concatenate([Xtr_imp, ind_sc.fit_transform(ind_tr).astype(np.float32)], axis=1)
            Xal_imp = np.concatenate([Xal_imp, ind_sc.transform(ind_al).astype(np.float32)], axis=1)

    n, d = Xtr_imp.shape
    n_comp = min(n_pca_components, max(1, n - 1), d)
    pca = CPU_PCA(n_components=n_comp, random_state=random_state)
    pca.fit(Xtr_imp)
    return pca.transform(Xal_imp).astype(np.float32)


def _prot_notmiwae_cache_path(
    seed: int,
    arm: str,
    fold_i: int,
    strat_name: str,
    missing_model: str,
    force_indicators: bool,
) -> Path:
    return _cache_file(
        OUT_CACHE,
        "prot_notmiwae",
        cache_tag=CACHE_TAG,
        seed=seed,
        arm=arm,
        fold_i=fold_i,
        strat_name=strat_name,
        missing_model=missing_model,
        force_indicators=force_indicators,
    )


def _prot_notmiwae_aux_cache_path(
    seed: int,
    arm: str,
    fold_i: int,
    strat_name: str,
    missing_model: str,
    force_indicators: bool,
    aux_name: str,
) -> Path:
    return _cache_file(
        OUT_CACHE,
        "prot_notmiwae_aux",
        cache_tag=CACHE_TAG,
        seed=seed,
        arm=arm,
        fold_i=fold_i,
        strat_name=strat_name,
        missing_model=missing_model,
        force_indicators=force_indicators,
        aux_name=aux_name,
    )


def load_or_build_prot_notmiwae_fold_cache(
    seed: int,
    arm: str,
    fold_i: int,
    strat_name: str,
    missing_model: str,
    add_indicators: bool,
    force_indicators: bool,
    prot_elig_values: np.ndarray,
    idx_train_full: np.ndarray,
) -> np.ndarray:
    path = _prot_notmiwae_cache_path(seed, arm, fold_i, strat_name, missing_model, force_indicators)
    if path.exists():
        return load(path)

    Xp = notmiwae_impute_block(
        X_train=prot_elig_values[idx_train_full],
        X_all=prot_elig_values,
        random_state=seed + 300,
        missing_model=missing_model,
        add_indicators=add_indicators,
        force_indicators=force_indicators,
        n_pca_components=PCA_COMPONENTS,
        aux_train=None,
        aux_all=None,
    )
    dump(Xp, path)
    return Xp


def load_or_build_prot_notmiwae_aux_fold_cache(
    seed: int,
    arm: str,
    fold_i: int,
    strat_name: str,
    missing_model: str,
    aux_name: str,
    add_indicators: bool,
    force_indicators: bool,
    prot_elig_values: np.ndarray,
    aux_all_values: np.ndarray,
    idx_train_full: np.ndarray,
) -> np.ndarray:
    path = _prot_notmiwae_aux_cache_path(
        seed=seed,
        arm=arm,
        fold_i=fold_i,
        strat_name=strat_name,
        missing_model=missing_model,
        force_indicators=force_indicators,
        aux_name=aux_name,
    )
    if path.exists():
        return load(path)

    Xp = notmiwae_impute_block(
        X_train=prot_elig_values[idx_train_full],
        X_all=prot_elig_values,
        random_state=seed + 330,
        missing_model=missing_model,
        add_indicators=add_indicators,
        force_indicators=force_indicators,
        n_pca_components=PCA_COMPONENTS,
        aux_train=aux_all_values[idx_train_full],
        aux_all=aux_all_values,
    )
    dump(Xp, path)
    return Xp


In [7]:
# Data load
cell_index = read_parquet_strict(IN_CLEAN / "cell_index.parquet")
prism_long = read_parquet_strict(IN_CLEAN / "prism_long.parquet")

rna = normalise_str_index(read_parquet_strict(IN_T2 / "rna.parquet"))
cnv = normalise_str_index(read_parquet_strict(IN_T2 / "cnv.parquet"))
mut = normalise_str_index(read_parquet_strict(IN_T2 / "mut.parquet"))

cell_index["depmap_id"] = cell_index["depmap_id"].astype(str).str.strip()
group_col = pick_group_column(cell_index)
groups_all = (
    cell_index.set_index("depmap_id")[group_col]
    .astype("string").fillna("Unknown").astype(str)
)

core_cells = sorted(set(rna.index) & set(cnv.index) & set(mut.index))
print(f"\nCore cohort (RNA ∩ CNV ∩ MUT): {len(core_cells)} cell lines")
print(f"Group column for CV: {group_col}")

prism_long["depmap_id"] = prism_long["depmap_id"].astype(str).str.strip()
prism_long["compound_id"] = prism_long["compound_id"].astype(str).str.strip()
prism_long["target"] = prism_long["target"].astype(str).str.strip().str.lower()

prism_auc = prism_long[prism_long["target"] == PRIMARY_TARGET][
    ["depmap_id", "compound_id", "y"]
].copy()

prot_arm_data: Dict[str, pd.DataFrame] = {}
for arm in ACTIVE_ARMS:
    p = IN_T2 / f"prot_optional__{arm}.parquet"
    if p.exists():
        prot_arm_data[arm] = normalise_str_index(pd.read_parquet(p))
        print(f"Loaded {arm}: {prot_arm_data[arm].shape}")
    else:
        print(f"[WARN] {arm} not found at {p}, skipping.")

drug_cov = prism_auc.groupby("compound_id")["depmap_id"].nunique().sort_values(ascending=False)
selected_drugs = drug_cov.head(N_DRUGS_BAKEOFF).index.tolist()
prism_sel = prism_auc[prism_auc["compound_id"].isin(selected_drugs)].copy()
drug_to_pairs = {k: v for k, v in prism_sel.groupby("compound_id", sort=False)}
print(f"\nSelected {len(selected_drugs)} drugs for not-MIWAE bake-off by PRISM AUC coverage.")


Core cohort (RNA ∩ CNV ∩ MUT): 1079 cell lines
Group column for CV: lineage_1
Loaded prot_combined_union: (1079, 18751)
Loaded prot_ms_ccle_gygi: (1079, 11780)
Loaded prot_procan_depmapSanger: (1079, 7906)
Loaded prot_rppa_ccle: (1079, 144)

Selected 100 drugs for not-MIWAE bake-off by PRISM AUC coverage.


In [8]:
# Missingness report
print("MISSINGNESS REPORT")

avail_rows = {"rna": {}, "cnv": {}, "mut": {}}
for c in core_cells:
    avail_rows["rna"][c] = 1 if c in rna.index else 0
    avail_rows["cnv"][c] = 1 if c in cnv.index else 0
    avail_rows["mut"][c] = 1 if c in mut.index else 0
for arm, df in prot_arm_data.items():
    avail_rows[arm] = df.reindex(core_cells).notna().any(axis=1).astype(int).to_dict()

availability = pd.DataFrame(avail_rows, index=core_cells)
availability.index.name = "depmap_id"
availability.to_csv(OUT_REPORTS / "modality_availability_matrix.csv")
print(f"Availability matrix: {OUT_REPORTS / 'modality_availability_matrix.csv'}  shape={availability.shape}")

avail_summary = (
    availability.sum().rename("n_cells_present").to_frame()
    .assign(
        n_cells_total=len(core_cells),
        pct_present=lambda d: (d["n_cells_present"] / len(core_cells) * 100).round(2),
    )
)
avail_summary.to_csv(OUT_REPORTS / "modality_availability_summary.csv")
print("\nModality availability summary:")
display(avail_summary)

feat_miss_rows = []
for arm, df in prot_arm_data.items():
    df_core = df.reindex(core_cells)
    col_miss = df_core.isna().mean()
    row_miss = df_core.isna().mean(axis=1)
    feat_miss_rows.append({
        "arm": arm,
        "n_features": int(df_core.shape[1]),
        "n_cells_in_core": int(df_core.shape[0]),
        "overall_missing_pct": float(df_core.isna().mean().mean() * 100),
        "col_miss_q10": float(col_miss.quantile(0.10)),
        "col_miss_q25": float(col_miss.quantile(0.25)),
        "col_miss_q50": float(col_miss.quantile(0.50)),
        "col_miss_q75": float(col_miss.quantile(0.75)),
        "col_miss_q90": float(col_miss.quantile(0.90)),
        "col_miss_q99": float(col_miss.quantile(0.99)),
        "row_miss_q10": float(row_miss.quantile(0.10)),
        "row_miss_q50": float(row_miss.quantile(0.50)),
        "row_miss_q90": float(row_miss.quantile(0.90)),
        "n_cols_fully_missing": int((col_miss == 1.0).sum()),
        "n_rows_fully_missing": int((row_miss == 1.0).sum()),
        "n_cols_zero_missing": int((col_miss == 0.0).sum()),
    })
feat_miss_df = pd.DataFrame(feat_miss_rows)
feat_miss_df.to_csv(OUT_REPORTS / "per_arm_feature_missingness.csv", index=False)
print("\nPer-arm feature missingness:")
display(feat_miss_df)

pat_counts = None
platform_miss_df = None
if TRACK2_ARM in prot_arm_data:
    union_df = prot_arm_data[TRACK2_ARM].reindex(core_cells)
    prefixes = {"ms": "ms__", "rppa": "rppa__", "procan": "procan__"}
    plat_pres = pd.DataFrame(index=union_df.index)
    for key, pref in prefixes.items():
        cols = [c for c in union_df.columns if str(c).startswith(pref)]
        plat_pres[key] = union_df[cols].notna().any(axis=1).astype(int) if cols else 0

    def pattern_label(row) -> str:
        parts = [k for k in ["ms", "rppa", "procan"] if row.get(k, 0) == 1]
        return "+".join(parts) if parts else "none"

    pat_counts = (
        plat_pres.apply(pattern_label, axis=1).rename("pattern")
        .value_counts().rename_axis("pattern").reset_index(name="n_cells")
    )
    pat_counts["frac_cells"] = pat_counts["n_cells"] / float(union_df.shape[0])
    pat_counts.to_csv(OUT_REPORTS / "combined_union_platform_patterns.csv", index=False)
    print("\nCombined union platform patterns:")
    display(pat_counts)

    pm_rows = []
    for key, pref in prefixes.items():
        cols = [c for c in union_df.columns if str(c).startswith(pref)]
        if not cols:
            continue
        n_absent = int(plat_pres[key].eq(0).sum())
        miss_from_abs = n_absent * len(cols)
        miss_total = int(union_df[cols].isna().sum().sum())
        pm_rows.append({
            "platform": key,
            "n_features": len(cols),
            "n_cells_present": int(plat_pres[key].sum()),
            "n_cells_absent": n_absent,
            "frac_cells_present": float(plat_pres[key].mean()),
            "pct_missingness_from_platform_absence": (
                float(miss_from_abs / miss_total * 100) if miss_total > 0 else np.nan
            ),
        })
    platform_miss_df = pd.DataFrame(pm_rows)
    platform_miss_df.to_csv(OUT_REPORTS / "combined_union_platform_missingness_contrib.csv", index=False)
    print("\nCombined union, missingness from platform absence:")
    display(platform_miss_df)


def build_missingness_report(
    avail_summary: pd.DataFrame,
    feat_miss_df: pd.DataFrame,
    pat_counts: Optional[pd.DataFrame],
    platform_miss_df: Optional[pd.DataFrame],
    track2_arm: str,
    all_seeds: List[int],
    ts: str,
) -> dict:
    report = {
        "generated_at": ts,
        "seeds": all_seeds,
        "active_arms": ACTIVE_ARMS,
        "deprioritised_arm": DEPRIORITISED_ARM,
        "leakage_note": (
            "All preprocessing, not-MIWAE fitting, PCA, and downstream models are "
            "fit inside CV folds only. No global statistics are used."
        ),
        "modality_availability": avail_summary.reset_index().to_dict(orient="records"),
        "per_arm_feature_missingness": feat_miss_df.to_dict(orient="records"),
        "notmiwae_settings": {
            "latent_dim": NOTMIWAE_LATENT_DIM,
            "hidden_dim": NOTMIWAE_HIDDEN_DIM,
            "train_k": NOTMIWAE_TRAIN_K,
            "impute_k": NOTMIWAE_IMPUTE_K,
            "epochs": NOTMIWAE_EPOCHS,
            "patience": NOTMIWAE_PATIENCE,
            "batch_size": NOTMIWAE_BATCH_SIZE,
        },
    }
    if pat_counts is not None:
        report["combined_union_platform_patterns"] = {
            "arm": track2_arm,
            "structural_missingness_warning": (
                "Combined-union missingness is heavily shaped by platform availability. "
                "Indicators remain compulsory for the combined-union arm."
            ),
            "patterns": pat_counts.to_dict(orient="records"),
        }
    if platform_miss_df is not None:
        report["combined_union_platform_missingness_contrib"] = platform_miss_df.to_dict(orient="records")
    report["bakeoff_outputs"] = {
        "detail_file": f"notmiwae_bakeoff_merged_{len(all_seeds)}seeds.csv",
        "summary_file": f"notmiwae_bakeoff_summary_merged_{len(all_seeds)}seeds.csv",
        "lock_file": NOTMIWAE_LOCK_FILENAME,
    }
    return report


missingness_report = build_missingness_report(
    avail_summary=avail_summary,
    feat_miss_df=feat_miss_df,
    pat_counts=pat_counts,
    platform_miss_df=platform_miss_df,
    track2_arm=TRACK2_ARM,
    all_seeds=ALL_SEEDS,
    ts=datetime.now(timezone.utc).isoformat(),
)
write_json(missingness_report, OUT_REPORTS / "missingness_report.json")
print(f"\nMissingness report written: {OUT_REPORTS / 'missingness_report.json'}")

# not-MIWAE proteomics bake-off

print("NOT-MIWAE PROTEOMICS BAKE-OFF (3 seeds, leakage-safe)")

all_bakeoff_rows: List[dict] = []
seeds_to_run: List[int] = []
REQUIRED_NEW_COLS = {"config_rank", "model", "feature_set", "uses_prot"}

EXPECTED_CONFIG_KEYS = {
    (int(cfg["rank"]), str(cfg["arm"]), str(cfg["model"]).lower(), str(cfg["feature_set"]))
    for cfg in NOTMIWAE_TEST_CONFIGS
}


def checkpoint_matches_expected_grid(df: pd.DataFrame) -> bool:
    needed = {"config_rank", "arm", "model", "feature_set"}
    if not needed.issubset(df.columns):
        return False
    found = {
        (int(r["config_rank"]), str(r["arm"]), str(r["model"]).lower(), str(r["feature_set"]))
        for _, r in df[["config_rank", "arm", "model", "feature_set"]].drop_duplicates().iterrows()
    }
    return found.issubset(EXPECTED_CONFIG_KEYS)


for run_seed in ALL_SEEDS:
    seed_path = OUT_REPORTS / f"notmiwae_bakeoff_seed{run_seed}.csv"
    if seed_path.exists():
        existing = pd.read_csv(seed_path)
        if (not REQUIRED_NEW_COLS.issubset(existing.columns)) or (not checkpoint_matches_expected_grid(existing)):
            print(f"[resume] Seed {run_seed} file exists but is from an old schema/grid, rerunning.")
            seeds_to_run.append(run_seed)
            continue
        existing["seed"] = existing["seed"].astype(int)
        n_drugs_in_file = existing["compound_id"].nunique()
        if n_drugs_in_file >= N_DRUGS_BAKEOFF:
            print(f"[resume] Seed {run_seed} complete ({n_drugs_in_file} drugs), loading.")
            all_bakeoff_rows.extend(existing.to_dict(orient="records"))
        else:
            print(f"[resume] Seed {run_seed} incomplete ({n_drugs_in_file}/{N_DRUGS_BAKEOFF} drugs), will rerun.")
            seeds_to_run.append(run_seed)
    else:
        seeds_to_run.append(run_seed)

if not seeds_to_run:
    print("[resume] All seeds complete, skipping not-MIWAE loop.")
else:
    print(f"[resume] Seeds remaining: {seeds_to_run}")

for run_seed in seeds_to_run:
    set_seeds(run_seed)
    print(f"  Seed {run_seed}")

    for arm in ACTIVE_ARMS:
        if arm not in prot_arm_data:
            print(f"  [SKIP] {arm} not loaded.")
            continue

        arm_cfgs = CONFIGS_BY_ARM.get(arm, [])
        if not arm_cfgs:
            print(f"  [SKIP] {arm} has no configs assigned.")
            continue

        prot_df = prot_arm_data[arm].copy()
        prot_df.index = prot_df.index.astype(str).str.strip()
        prot_core = prot_df.reindex(core_cells)

        has_prot = prot_core.notna().any(axis=1)
        eligible_cells = sorted(has_prot[has_prot].index.tolist())
        if len(eligible_cells) < 200:
            print(f"  [SKIP] {arm}: only {len(eligible_cells)} eligible cells.")
            continue

        arm_ckpt_path = OUT_REPORTS / f"notmiwae_bakeoff_seed{run_seed}_{arm}.csv"
        already_done_drugs: set = set()

        if arm_ckpt_path.exists():
            arm_existing = pd.read_csv(arm_ckpt_path)
            if REQUIRED_NEW_COLS.issubset(arm_existing.columns) and checkpoint_matches_expected_grid(arm_existing):
                arm_existing["seed"] = arm_existing["seed"].astype(int)
                n_drugs_in_arm = arm_existing["compound_id"].nunique()
                if n_drugs_in_arm >= N_DRUGS_BAKEOFF:
                    print(f"  [resume] seed={run_seed} arm={arm} complete ({n_drugs_in_arm} drugs), loading.")
                    all_bakeoff_rows.extend(arm_existing.to_dict(orient="records"))
                    continue
                else:
                    print(f"  [resume] seed={run_seed} arm={arm} partial ({n_drugs_in_arm}/{N_DRUGS_BAKEOFF} drugs), resuming.")
                    all_bakeoff_rows.extend(arm_existing.to_dict(orient="records"))
                    already_done_drugs = set(arm_existing["compound_id"].unique().tolist())
            else:
                print(f"  [resume] seed={run_seed} arm={arm} checkpoint is old schema, rerunning arm.")

        arm_groups = groups_all.reindex(eligible_cells).fillna("Unknown").astype(str)
        splits, split_name = safe_group_splits(eligible_cells, arm_groups, N_SPLITS_DESIRED, seed=run_seed)
        print(f"  {arm}: {split_name}, {len(eligible_cells)} cells")

        fold_map = {}
        for fold_i, (_, test_idx) in enumerate(splits):
            for j in test_idx:
                fold_map[eligible_cells[int(j)]] = fold_i

        prot_elig = prot_core.loc[eligible_cells]
        prot_elig_values = prot_elig.to_numpy(dtype=float)
        fold_cache: Dict[int, dict] = {}
        force_indicators = (arm == TRACK2_ARM)

        print(f"    Building/loading not-MIWAE fold caches for {arm}...")
        for fold_i, (train_idx, _) in enumerate(splits):
            train_cells = [eligible_cells[int(j)] for j in train_idx]
            base_payload = load_or_build_base_fold_cache(
                seed=run_seed,
                arm=arm,
                fold_i=fold_i,
                train_cells=train_cells,
                eligible_cells=eligible_cells,
                rna_df=rna,
                cnv_df=cnv,
                mut_df=mut,
            )

            X_aux_all = np.concatenate([base_payload["Xr"], base_payload["Xc"], base_payload["Xm"]], axis=1)
            fold_entry = {
                "base_mats": {
                    "rna": base_payload["Xr"],
                    "cnv": base_payload["Xc"],
                    "mut": base_payload["Xm"],
                },
                "prot": {},
                "prot_aux": {},
            }

            for strat in NOTMIWAE_STRATEGIES:
                strat_name = str(strat["name"])
                add_ind = bool(strat["add_indicators"])
                missing_model = str(strat["missing_model"])

                try:
                    Xp = load_or_build_prot_notmiwae_fold_cache(
                        seed=run_seed,
                        arm=arm,
                        fold_i=fold_i,
                        strat_name=strat_name,
                        missing_model=missing_model,
                        add_indicators=add_ind,
                        force_indicators=force_indicators,
                        prot_elig_values=prot_elig_values,
                        idx_train_full=np.asarray(train_idx, dtype=int),
                    )
                except Exception as e:
                    print(f"    [WARN] not-MIWAE cache build failed {arm}/{strat_name}/fold{fold_i}: {e}")
                    Xp = None
                fold_entry["prot"][(strat_name, add_ind, force_indicators)] = Xp

                if USE_AUX_FOR_PROT_IMPUTATION and missing_model in {"linear", "nonlinear"}:
                    try:
                        Xp_aux = load_or_build_prot_notmiwae_aux_fold_cache(
                            seed=run_seed,
                            arm=arm,
                            fold_i=fold_i,
                            strat_name=strat_name,
                            missing_model=missing_model,
                            aux_name=AUX_FOR_PROT_NAME,
                            add_indicators=add_ind,
                            force_indicators=force_indicators,
                            prot_elig_values=prot_elig_values,
                            aux_all_values=X_aux_all,
                            idx_train_full=np.asarray(train_idx, dtype=int),
                        )
                    except Exception as e:
                        print(f"    [WARN] auxiliary not-MIWAE cache build failed {arm}/{strat_name}/fold{fold_i}: {e}")
                        Xp_aux = None
                    fold_entry["prot_aux"][(strat_name, add_ind, force_indicators)] = Xp_aux

            fold_cache[fold_i] = fold_entry

        n_run = 0
        n_skip = 0

        for drug in selected_drugs:
            if drug in already_done_drugs:
                continue

            pairs = drug_to_pairs.get(drug)
            if pairs is None:
                n_skip += 1
                continue

            df = pairs[pairs["depmap_id"].isin(eligible_cells)][["depmap_id", "y"]].copy()
            if df["depmap_id"].nunique() < MIN_CELLS_PER_DRUG:
                n_skip += 1
                continue

            df = df.groupby("depmap_id", as_index=False)["y"].mean()
            cell_ids = df["depmap_id"].astype(str).tolist()
            y_all = df["y"].to_numpy(dtype=float)

            fold_ids = np.array([fold_map.get(c, -1) for c in cell_ids], dtype=int)
            valid = fold_ids >= 0
            cell_ids = [c for c, v in zip(cell_ids, valid) if v]
            y_all = y_all[valid]
            fold_ids = fold_ids[valid]

            if len(cell_ids) < MIN_CELLS_PER_DRUG:
                n_skip += 1
                continue

            n_run += 1
            if n_run % 5 == 0:
                arm_rows_partial = [r for r in all_bakeoff_rows if r["seed"] == run_seed and r["arm"] == arm]
                if arm_rows_partial:
                    arm_df_partial = pd.DataFrame(arm_rows_partial)
                    arm_df_partial.to_csv(arm_ckpt_path, index=False)
                    cumulative_done = arm_df_partial["compound_id"].nunique()
                    print(f"    [mid-checkpoint] drug {cumulative_done}/{N_DRUGS_BAKEOFF} | {arm_ckpt_path}")

            c2r = {c: i for i, c in enumerate(eligible_cells)}
            idx_all = np.array([c2r[c] for c in cell_ids], dtype=int)

            for fold_i, _ in enumerate(splits):
                in_test = fold_ids == fold_i
                n_test = int(in_test.sum())
                n_train = int((~in_test).sum())
                if n_test < 5 or n_train < 20:
                    continue

                idx_train = idx_all[~in_test]
                idx_test = idx_all[in_test]
                y_train = y_all[~in_test]
                y_test = y_all[in_test]

                base_mats = fold_cache[fold_i]["base_mats"]

                for cfg in arm_cfgs:
                    cfg_rank = int(cfg["rank"])
                    cfg_model = str(cfg["model"]).lower()
                    cfg_feature_set = str(cfg["feature_set"])
                    cfg_keys = parse_feature_set(cfg_feature_set)
                    uses_prot = "prot" in cfg_keys

                    X_nonprot = concat_selected_modalities(
                        mats=base_mats,
                        selected_keys=tuple(k for k in cfg_keys if k != "prot"),
                        n_rows=len(eligible_cells),
                    )

                    if not uses_prot:
                        if X_nonprot.shape[1] == 0:
                            continue
                        row = load_or_run_eval_cache(
                            kind="notmiwae_eval",
                            seed=run_seed,
                            arm=arm,
                            drug=drug,
                            fold_i=fold_i,
                            cfg_rank=cfg_rank,
                            model_name=cfg_model,
                            feature_set=cfg_feature_set,
                            imputer_strategy="no_prot_reference",
                            extra_meta={
                                "uses_prot": False,
                                "add_indicators": False,
                                "force_indicators": False,
                            },
                            X_train=X_nonprot[idx_train],
                            X_test=X_nonprot[idx_test],
                            y_train=y_train,
                            y_test=y_test,
                        )
                        all_bakeoff_rows.append(row)
                        continue

                    cfg_keys_wo_prot = tuple(k for k in cfg_keys if k != "prot")
                    if len(cfg_keys_wo_prot) > 0:
                        X_ref = concat_selected_modalities(base_mats, cfg_keys_wo_prot, len(eligible_cells))
                        if X_ref.shape[1] > 0:
                            row = load_or_run_eval_cache(
                                kind="notmiwae_eval",
                                seed=run_seed,
                                arm=arm,
                                drug=drug,
                                fold_i=fold_i,
                                cfg_rank=cfg_rank,
                                model_name=cfg_model,
                                feature_set=cfg_feature_set,
                                imputer_strategy="reference_without_prot",
                                extra_meta={
                                    "uses_prot": True,
                                    "add_indicators": False,
                                    "force_indicators": False,
                                },
                                X_train=X_ref[idx_train],
                                X_test=X_ref[idx_test],
                                y_train=y_train,
                                y_test=y_test,
                            )
                            all_bakeoff_rows.append(row)

                    for strat in NOTMIWAE_STRATEGIES:
                        strat_name = str(strat["name"])
                        add_ind = bool(strat["add_indicators"])
                        missing_model = str(strat["missing_model"])

                        Xp = fold_cache[fold_i]["prot"].get((strat_name, add_ind, force_indicators))
                        if Xp is not None and Xp.shape[1] > 0:
                            parts = []
                            bad_cfg = False
                            for k in cfg_keys:
                                if k == "prot":
                                    parts.append(Xp)
                                else:
                                    arr = base_mats.get(k)
                                    if arr is None or arr.shape[1] == 0:
                                        bad_cfg = True
                                        break
                                    parts.append(arr)
                            if not bad_cfg and len(parts) > 0:
                                Xf = np.concatenate(parts, axis=1)
                                row = load_or_run_eval_cache(
                                    kind="notmiwae_eval",
                                    seed=run_seed,
                                    arm=arm,
                                    drug=drug,
                                    fold_i=fold_i,
                                    cfg_rank=cfg_rank,
                                    model_name=cfg_model,
                                    feature_set=cfg_feature_set,
                                    imputer_strategy=strat_name,
                                    extra_meta={
                                        "uses_prot": True,
                                        "add_indicators": add_ind,
                                        "force_indicators": force_indicators,
                                        "missing_model": missing_model,
                                    },
                                    X_train=Xf[idx_train],
                                    X_test=Xf[idx_test],
                                    y_train=y_train,
                                    y_test=y_test,
                                )
                                all_bakeoff_rows.append(row)

                        if USE_AUX_FOR_PROT_IMPUTATION and missing_model in {"linear", "nonlinear"}:
                            Xp_aux = fold_cache[fold_i]["prot_aux"].get((strat_name, add_ind, force_indicators))
                            if Xp_aux is not None and Xp_aux.shape[1] > 0:
                                parts_aux = []
                                bad_cfg_aux = False
                                for k in cfg_keys:
                                    if k == "prot":
                                        parts_aux.append(Xp_aux)
                                    else:
                                        arr = base_mats.get(k)
                                        if arr is None or arr.shape[1] == 0:
                                            bad_cfg_aux = True
                                            break
                                        parts_aux.append(arr)
                                if not bad_cfg_aux and len(parts_aux) > 0:
                                    Xf_aux = np.concatenate(parts_aux, axis=1)
                                    imp_name = f"{AUX_FOR_PROT_NAME}::{strat_name}"
                                    row = load_or_run_eval_cache(
                                        kind="notmiwae_eval",
                                        seed=run_seed,
                                        arm=arm,
                                        drug=drug,
                                        fold_i=fold_i,
                                        cfg_rank=cfg_rank,
                                        model_name=cfg_model,
                                        feature_set=cfg_feature_set,
                                        imputer_strategy=imp_name,
                                        extra_meta={
                                            "uses_prot": True,
                                            "add_indicators": add_ind,
                                            "force_indicators": force_indicators,
                                            "missing_model": missing_model,
                                            "auxiliary_conditioning": True,
                                        },
                                        X_train=Xf_aux[idx_train],
                                        X_test=Xf_aux[idx_test],
                                        y_train=y_train,
                                        y_test=y_test,
                                    )
                                    all_bakeoff_rows.append(row)

        print(f"    drugs_run={n_run}, drugs_skipped={n_skip}")
        arm_rows = [r for r in all_bakeoff_rows if r["seed"] == run_seed and r["arm"] == arm]
        arm_df = pd.DataFrame(arm_rows)
        arm_df.to_csv(arm_ckpt_path, index=False)
        print(f"    [checkpoint] {arm_ckpt_path}  shape={arm_df.shape}")

    seed_df = pd.DataFrame([r for r in all_bakeoff_rows if r["seed"] == run_seed])
    seed_path = OUT_REPORTS / f"notmiwae_bakeoff_seed{run_seed}.csv"
    seed_df.to_csv(seed_path, index=False)
    print(f"  Saved seed {run_seed}: {seed_path}  shape={seed_df.shape}")

MISSINGNESS REPORT
Availability matrix: artifacts/reports/notebook 7_notmiwae_prot/modality_availability_matrix.csv  shape=(1079, 7)

Modality availability summary:


,n_cells_present,n_cells_total,pct_present
rna,1079,1079,100.00
cnv,1079,1079,100.00
mut,1079,1079,100.00
prot_combined_union,679,1079,62.93
prot_ms_ccle_gygi,304,1079,28.17
prot_procan_depmapSanger,485,1079,44.95
prot_rppa_ccle,612,1079,56.72



Per-arm feature missingness:


,arm,n_features,n_cells_in_core,overall_missing_pct,col_miss_q10,col_miss_q25,col_miss_q50,col_miss_q75,col_miss_q90,col_miss_q99,row_miss_q10,row_miss_q50,row_miss_q90,n_cols_fully_missing,n_rows_fully_missing,n_cols_zero_missing
0,prot_combined_union,18751,1079,74.118180,0.556070,0.718258,0.718258,0.825765,0.915663,0.961075,0.249363,0.760706,1.0,0,400,0
1,prot_ms_ccle_gygi,11780,1079,78.876396,0.718258,0.718258,0.732159,0.861214,0.949027,0.979611,0.237267,1.000000,1.0,0,775,0
2,prot_procan_depmapSanger,7906,1079,70.717562,0.550510,0.560704,0.668211,0.843373,0.931418,0.975904,0.308981,1.000000,1.0,0,594,0
3,prot_rppa_ccle,144,1079,43.280816,0.432808,0.432808,0.432808,0.432808,0.432808,0.432808,0.000000,0.000000,1.0,0,467,0



Combined union platform patterns:


,pattern,n_cells,frac_cells
0,none,400,0.370714
1,ms+rppa+procan,233,0.215941
2,rppa+procan,187,0.173309
3,rppa,128,0.118628
4,ms+rppa,64,0.059314
5,procan,60,0.055607
6,ms+procan,5,0.004634
7,ms,2,0.001854



Combined union, missingness from platform absence:


,platform,n_features,n_cells_present,n_cells_absent,frac_cells_present,pct_missingness_from_platform_absence
0,ms,10847,304,775,0.281742,92.892390
1,rppa,144,612,467,0.567192,100.000000
2,procan,7760,485,594,0.449490,78.405864



Missingness report written: artifacts/reports/notebook 7_notmiwae_prot/missingness_report.json
NOT-MIWAE PROTEOMICS BAKE-OFF (3 seeds, leakage-safe)
[resume] Seed 19537 complete (100 drugs), loading.
[resume] Seed 1584678 complete (100 drugs), loading.
[resume] Seeds remaining: [17052356]
  Seed 17052356
  [resume] seed=17052356 arm=prot_combined_union partial (34/100 drugs), resuming.
  prot_combined_union: GroupKFold(n_splits=10), 679 cells
    Building/loading not-MIWAE fold caches for prot_combined_union...
    [mid-checkpoint] drug 38/100 | artifacts/reports/notebook 7_notmiwae_prot/notmiwae_bakeoff_seed17052356_prot_combined_union.csv
    [mid-checkpoint] drug 43/100 | artifacts/reports/notebook 7_notmiwae_prot/notmiwae_bakeoff_seed17052356_prot_combined_union.csv
    [mid-checkpoint] drug 48/100 | artifacts/reports/notebook 7_notmiwae_prot/notmiwae_bakeoff_seed17052356_prot_combined_union.csv
    [mid-checkpoint] drug 53/100 | artifacts/reports/notebook 7_notmiwae_prot/notmiwae

In [9]:
# Summaries

bakeoff_df = pd.DataFrame(all_bakeoff_rows)
merged_path = OUT_REPORTS / f"notmiwae_bakeoff_merged_{len(ALL_SEEDS)}seeds.csv"
bakeoff_df.to_csv(merged_path, index=False)
print(f"\nMerged not-MIWAE bake-off: {merged_path}  shape={bakeoff_df.shape}")

drug_means = (
    bakeoff_df
    .groupby(
        ["seed", "config_rank", "arm", "model", "feature_set", "uses_prot", "imputer_strategy", "compound_id"],
        as_index=False,
    )
    .agg(
        spearman=("spearman", safe_nanmean),
        r2=("r2", safe_nanmean),
        n_folds=("fold", "nunique"),
    )
)

bakeoff_summary = (
    drug_means
    .groupby(
        ["config_rank", "arm", "model", "feature_set", "uses_prot", "imputer_strategy"],
        as_index=False,
    )
    .agg(
        n_seeds=("seed", "nunique"),
        n_drugs=("compound_id", "nunique"),
        mean_spearman=("spearman", safe_nanmean),
        median_spearman=("spearman", safe_nanmedian),
        std_spearman=("spearman", safe_nanstd),
        mean_r2=("r2", safe_nanmean),
    )
    .sort_values(["config_rank", "mean_spearman"], ascending=[True, False])
)

base_ref = (
    bakeoff_summary[
        bakeoff_summary["imputer_strategy"].isin(["no_prot_reference", "reference_without_prot"])
    ][["config_rank", "arm", "model", "feature_set", "mean_spearman", "imputer_strategy"]]
    .rename(columns={"mean_spearman": "baseline_mean_spearman", "imputer_strategy": "baseline_strategy"})
    .drop_duplicates(subset=["config_rank", "arm", "model", "feature_set"])
)

bakeoff_summary = bakeoff_summary.merge(
    base_ref,
    on=["config_rank", "arm", "model", "feature_set"],
    how="left",
)

bakeoff_summary["delta_vs_baseline"] = np.where(
    bakeoff_summary["imputer_strategy"].isin(["no_prot_reference", "reference_without_prot"]),
    0.0,
    bakeoff_summary["mean_spearman"] - bakeoff_summary["baseline_mean_spearman"],
)

summary_path = OUT_REPORTS / f"notmiwae_bakeoff_summary_merged_{len(ALL_SEEDS)}seeds.csv"
bakeoff_summary.to_csv(summary_path, index=False)
print("\nnot-MIWAE bake-off summary:")
display(bakeoff_summary)

per_seed_summary = (
    drug_means
    .groupby(
        ["seed", "config_rank", "arm", "model", "feature_set", "uses_prot", "imputer_strategy"],
        as_index=False,
    )
    .agg(
        n_drugs=("compound_id", "nunique"),
        mean_spearman=("spearman", safe_nanmean),
        median_spearman=("spearman", safe_nanmedian),
        std_spearman=("spearman", safe_nanstd),
        mean_r2=("r2", safe_nanmean),
    )
    .sort_values(["config_rank", "seed", "mean_spearman"], ascending=[True, True, False])
)

per_seed_path = OUT_REPORTS / "notmiwae_bakeoff_per_seed_summary.csv"
per_seed_summary.to_csv(per_seed_path, index=False)
print("\nPer-seed not-MIWAE summary:")
display(per_seed_summary)

# Lock decision
print("NOT-MIWAE LOCK DECISION")

notmiwae_choice = {}
for cfg in NOTMIWAE_TEST_CONFIGS:
    cfg_rank = int(cfg["rank"])
    cfg_arm = str(cfg["arm"])
    cfg_model = str(cfg["model"]).lower()
    cfg_feature_set = str(cfg["feature_set"])
    cfg_uses_prot = "prot" in parse_feature_set(cfg_feature_set)

    key = f"rank{cfg_rank}__{cfg_arm}__{cfg_model}__{cfg_feature_set}"

    cfg_df = bakeoff_summary[
        (bakeoff_summary["config_rank"] == cfg_rank) &
        (bakeoff_summary["arm"] == cfg_arm) &
        (bakeoff_summary["model"] == cfg_model) &
        (bakeoff_summary["feature_set"] == cfg_feature_set)
    ].copy()

    if cfg_df.shape[0] == 0:
        notmiwae_choice[key] = {
            "chosen_strategy": None,
            "reason": "No not-MIWAE bake-off rows produced for this config.",
        }
        continue

    if not cfg_uses_prot:
        ref_rows = cfg_df[cfg_df["imputer_strategy"] == "no_prot_reference"]
        ref_sp = float(ref_rows["mean_spearman"].iloc[0]) if len(ref_rows) else np.nan
        notmiwae_choice[key] = {
            "config_rank": cfg_rank,
            "arm": cfg_arm,
            "model": cfg_model,
            "feature_set": cfg_feature_set,
            "chosen_strategy": "not_applicable",
            "mean_spearman_chosen": round(ref_sp, 6) if np.isfinite(ref_sp) else None,
            "mean_spearman_baseline": round(ref_sp, 6) if np.isfinite(ref_sp) else None,
            "delta_vs_baseline": 0.0,
            "n_seeds_in_estimate": len(ALL_SEEDS),
            "reason": "This ranked configuration does not contain proteomics, so not-MIWAE imputation is not applicable.",
        }
        print(f"\n{key}:")
        print("  Chosen   : not_applicable")
        continue

    candidates = cfg_df[
        ~cfg_df["imputer_strategy"].isin(["no_prot_reference", "reference_without_prot"])
    ].copy()

    if cfg_arm == TRACK2_ARM:
        ind_candidates = candidates[candidates["imputer_strategy"].str.contains("indicator", regex=False)]
        if ind_candidates.shape[0] > 0:
            candidates = ind_candidates

    if candidates.shape[0] == 0:
        notmiwae_choice[key] = {
            "config_rank": cfg_rank,
            "arm": cfg_arm,
            "model": cfg_model,
            "feature_set": cfg_feature_set,
            "chosen_strategy": None,
            "reason": "No not-MIWAE candidates available for this config.",
        }
        continue

    best = candidates.sort_values("mean_spearman", ascending=False).iloc[0]
    base_rows = cfg_df[cfg_df["imputer_strategy"] == "reference_without_prot"]
    base_sp = float(base_rows["mean_spearman"].iloc[0]) if len(base_rows) else np.nan

    chosen = str(best["imputer_strategy"])
    mean_sp = float(best["mean_spearman"])
    std_sp = float(best["std_spearman"])
    delta = float(best["delta_vs_baseline"]) if pd.notna(best["delta_vs_baseline"]) else np.nan

    notmiwae_choice[key] = {
        "config_rank": cfg_rank,
        "arm": cfg_arm,
        "model": cfg_model,
        "feature_set": cfg_feature_set,
        "chosen_strategy": chosen,
        "mean_spearman_chosen": round(mean_sp, 6),
        "std_spearman_chosen": round(std_sp, 6),
        "mean_spearman_baseline": round(base_sp, 6) if np.isfinite(base_sp) else None,
        "delta_vs_baseline": round(delta, 6) if np.isfinite(delta) else None,
        "n_seeds_in_estimate": len(ALL_SEEDS),
        "reason": (
            f"Best mean Spearman ({mean_sp:.4f} ± {std_sp:.4f}) across {len(ALL_SEEDS)} seeds"
            + (
                f"; delta vs config-specific no-prot reference: {delta:+.4f}."
                if np.isfinite(delta) else
                "; no config-specific no-prot reference available."
            )
            + (" Indicators mandatory for combined union arm." if cfg_arm == TRACK2_ARM else "")
        ),
    }

    print(f"\n{key}:")
    print(f"  Chosen   : {chosen}")
    print(f"  Spearman : {mean_sp:.4f} ± {std_sp:.4f}")
    if np.isfinite(base_sp):
        print(f"  Baseline : {base_sp:.4f}  delta={delta:+.4f}")

notmiwae_choice_doc = {
    "locked_at": datetime.now(timezone.utc).isoformat(),
    "seeds_used": ALL_SEEDS,
    "n_drugs_bakeoff": N_DRUGS_BAKEOFF,
    "primary_target": PRIMARY_TARGET,
    "pca_components": PCA_COMPONENTS,
    "torch_device": str(TORCH_DEVICE),
    "notmiwae_settings": {
        "latent_dim": NOTMIWAE_LATENT_DIM,
        "hidden_dim": NOTMIWAE_HIDDEN_DIM,
        "dropout": NOTMIWAE_DROPOUT,
        "epochs": NOTMIWAE_EPOCHS,
        "patience": NOTMIWAE_PATIENCE,
        "batch_size": NOTMIWAE_BATCH_SIZE,
        "lr": NOTMIWAE_LR,
        "weight_decay": NOTMIWAE_WEIGHT_DECAY,
        "train_k": NOTMIWAE_TRAIN_K,
        "impute_k": NOTMIWAE_IMPUTE_K,
        "min_logvar": NOTMIWAE_MIN_LOGVAR,
        "max_logvar": NOTMIWAE_MAX_LOGVAR,
    },
    "test_configs": NOTMIWAE_TEST_CONFIGS,
    "note": (
        "This notebook uses a fold-safe not-MIWAE-style MNAR generative imputer for sparse proteomics. "
        "The model jointly scores observed proteomics values and the missingness mask. "
        "Self-masking and linear missing models are compared, with optional RNA/CNV/MUT auxiliary conditioning. "
        "All transforms, not-MIWAE fitting, PCA, and downstream regressors are fit inside training folds only."
    ),
    "config_choices": notmiwae_choice,
}

notmiwae_choice_path = OUT_META / NOTMIWAE_LOCK_FILENAME
write_json(notmiwae_choice_doc, notmiwae_choice_path)
print(f"\nnot-MIWAE imputer choice written: {notmiwae_choice_path}")

global_copy = IN_META / NOTMIWAE_LOCK_FILENAME
write_json(notmiwae_choice_doc, global_copy)
print(f"Global copy written: {global_copy}")

INTERP_SEED = ALL_SEEDS[0]
INTERP_ARM = PRIMARY_ARM
INTERP_MODEL = "extratrees"
INTERP_FEATURE_SET = "rna+prot"
INTERP_DRUG = None  # set a specific compound_id to override auto-selection
INTERP_FOLD = 0
TOP_N_FEATURES = 50


def choose_interpretability_target(
    detail_df: pd.DataFrame,
    summary_df: pd.DataFrame,
    arm: str,
    model: str,
    feature_set: str,
    seed: int,
    chosen_strategy_doc: dict,
) -> Tuple[str, str]:
    cfg_rows = summary_df[
        (summary_df["arm"] == arm) &
        (summary_df["model"] == model) &
        (summary_df["feature_set"] == feature_set)
    ].copy()
    if cfg_rows.empty:
        raise ValueError("No summary rows found for requested interpretability target.")

    best_summary = cfg_rows.sort_values("mean_spearman", ascending=False).iloc[0]
    chosen_strategy = str(best_summary["imputer_strategy"])

    detail_rows = detail_df[
        (detail_df["seed"] == seed) &
        (detail_df["arm"] == arm) &
        (detail_df["model"] == model) &
        (detail_df["feature_set"] == feature_set) &
        (detail_df["imputer_strategy"] == chosen_strategy)
    ].copy()
    if detail_rows.empty:
        raise ValueError("No detail rows found for requested interpretability target.")

    drug_rank = (
        detail_rows.groupby("compound_id", as_index=False)["spearman"]
        .mean()
        .sort_values("spearman", ascending=False)
    )
    chosen_drug = str(drug_rank.iloc[0]["compound_id"])
    return chosen_strategy, chosen_drug


def build_nonpca_original_block(
    df: pd.DataFrame,
    train_cells: List[str],
    all_cells: List[str],
    strategy: str = "median",
    do_scale: bool = True,
) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    X_train = df.loc[train_cells].to_numpy(dtype=float)
    X_all = df.loc[all_cells].to_numpy(dtype=float)
    keep = np.isfinite(X_train).any(axis=0)
    cols = df.columns[keep].astype(str).tolist()
    if keep.sum() == 0:
        return np.zeros((len(train_cells), 0), dtype=np.float32), np.zeros((len(all_cells), 0), dtype=np.float32), []
    X_train = X_train[:, keep]
    X_all = X_all[:, keep]
    imp = SimpleImputer(strategy=strategy)
    X_train_imp = imp.fit_transform(X_train)
    X_all_imp = imp.transform(X_all)
    if do_scale:
        sc = StandardScaler()
        X_train_imp = sc.fit_transform(X_train_imp)
        X_all_imp = sc.transform(X_all_imp)
    return X_train_imp.astype(np.float32), X_all_imp.astype(np.float32), cols


def build_notmiwae_original_prot_block(
    prot_df: pd.DataFrame,
    train_cells: List[str],
    all_cells: List[str],
    missing_model: str,
    aux_train: Optional[np.ndarray],
    aux_all: Optional[np.ndarray],
) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    scaler = StandardiseObservedBlock().fit(prot_df.loc[train_cells].to_numpy(dtype=float))
    Xtr_std, miss_tr = scaler.transform(prot_df.loc[train_cells].to_numpy(dtype=float))
    Xal_std, miss_al = scaler.transform(prot_df.loc[all_cells].to_numpy(dtype=float))

    if Xtr_std.shape[1] == 0:
        return np.zeros((len(train_cells), 0), dtype=np.float32), np.zeros((len(all_cells), 0), dtype=np.float32), []

    model = NotMIWAEImputer(
        latent_dim=NOTMIWAE_LATENT_DIM,
        hidden_dim=NOTMIWAE_HIDDEN_DIM,
        missing_model=missing_model,
        dropout=NOTMIWAE_DROPOUT,
        train_k=NOTMIWAE_TRAIN_K,
        impute_k=NOTMIWAE_IMPUTE_K,
        lr=NOTMIWAE_LR,
        weight_decay=NOTMIWAE_WEIGHT_DECAY,
        epochs=NOTMIWAE_EPOCHS,
        patience=NOTMIWAE_PATIENCE,
        batch_size=NOTMIWAE_BATCH_SIZE,
        min_logvar=NOTMIWAE_MIN_LOGVAR,
        max_logvar=NOTMIWAE_MAX_LOGVAR,
        device=TORCH_DEVICE,
    ).fit(X_train_std=Xtr_std, miss_train=miss_tr, seed=INTERP_SEED, aux_train=aux_train)

    Xtr_imp = model.impute(Xtr_std, miss_tr, aux=aux_train)
    Xal_imp = model.impute(Xal_std, miss_al, aux=aux_all)
    cols = prot_df.columns[scaler.keep_].astype(str).tolist()
    return Xtr_imp.astype(np.float32), Xal_imp.astype(np.float32), cols


def try_run_pathway_enrichment(feature_names: List[str], out_dir: Path) -> None:
    genes = [str(g).split("__")[-1] for g in feature_names if isinstance(g, str)]
    genes = [g for g in genes if g and not g.startswith("missing_")]
    genes = list(dict.fromkeys(genes))[:500]
    if len(genes) == 0:
        print("[pathway] No gene-like features available for enrichment.")
        return

    out_dir.mkdir(parents=True, exist_ok=True)

    try:
        gp = GProfiler(return_dataframe=True)
        enr = gp.profile(organism="hsapiens", query=genes)
        enr.to_csv(out_dir / "pathway_enrichment_gprofiler.csv", index=False)
        print(f"[pathway] g:Profiler results written to {out_dir / 'pathway_enrichment_gprofiler.csv'}")
        return
    except Exception as e:
        print(f"[pathway] g:Profiler unavailable: {e}")

    try:
        enr = gp.enrichr(
            gene_list=genes,
            gene_sets=["Reactome_2022", "KEGG_2021_Human", "GO_Biological_Process_2023"],
            organism="Human",
            outdir=str(out_dir / "gseapy"),
            cutoff=0.5,
        )
        if hasattr(enr, "results") and enr.results is not None:
            enr.results.to_csv(out_dir / "pathway_enrichment_gseapy.csv", index=False)
            print(f"[pathway] gseapy results written to {out_dir / 'pathway_enrichment_gseapy.csv'}")
            return
    except Exception as e:
        print(f"[pathway] gseapy unavailable: {e}")

    print("[pathway] No enrichment backend available. Install gprofiler-official or gseapy.")


if not bakeoff_df.empty:
    chosen_strategy, auto_drug = choose_interpretability_target(
        detail_df=bakeoff_df,
        summary_df=bakeoff_summary,
        arm=INTERP_ARM,
        model=INTERP_MODEL,
        feature_set=INTERP_FEATURE_SET,
        seed=INTERP_SEED,
        chosen_strategy_doc=notmiwae_choice_doc,
    )
    if INTERP_DRUG is None:
        INTERP_DRUG = auto_drug

    print("\nINTERPRETABILITY TARGET")
    print("  seed          :", INTERP_SEED)
    print("  arm           :", INTERP_ARM)
    print("  model         :", INTERP_MODEL)
    print("  feature_set   :", INTERP_FEATURE_SET)
    print("  strategy      :", chosen_strategy)
    print("  compound_id   :", INTERP_DRUG)
    print("  fold          :", INTERP_FOLD)

    prot_df = prot_arm_data[INTERP_ARM].copy()
    prot_df.index = prot_df.index.astype(str).str.strip()
    prot_core = prot_df.reindex(core_cells)
    eligible_cells = sorted(prot_core.notna().any(axis=1)[lambda s: s].index.tolist())
    arm_groups = groups_all.reindex(eligible_cells).fillna("Unknown").astype(str)
    splits, _ = safe_group_splits(eligible_cells, arm_groups, N_SPLITS_DESIRED, seed=INTERP_SEED)
    train_idx, test_idx = splits[INTERP_FOLD]
    train_cells = [eligible_cells[int(i)] for i in train_idx]
    test_cells = [eligible_cells[int(i)] for i in test_idx]

    base_payload = load_or_build_base_fold_cache(
        seed=INTERP_SEED,
        arm=INTERP_ARM,
        fold_i=INTERP_FOLD,
        train_cells=train_cells,
        eligible_cells=eligible_cells,
        rna_df=rna,
        cnv_df=cnv,
        mut_df=mut,
    )
    aux_all = np.concatenate([base_payload["Xr"], base_payload["Xc"], base_payload["Xm"]], axis=1)
    cell_to_row = {c: i for i, c in enumerate(eligible_cells)}
    aux_train = aux_all[[cell_to_row[c] for c in train_cells]] if "aux_rna_cnv_mut" in chosen_strategy else None
    aux_full = aux_all if "aux_rna_cnv_mut" in chosen_strategy else None

    if "selfmask" in chosen_strategy:
        interp_missing_model = "selfmask"
    elif "linear" in chosen_strategy:
        interp_missing_model = "linear"
    else:
        interp_missing_model = "linear"

    feat_keys = parse_feature_set(INTERP_FEATURE_SET)
    X_train_parts = []
    X_all_parts = []
    feature_names: List[str] = []

    if "rna" in feat_keys:
        Xr_tr, Xr_all, rna_cols = build_nonpca_original_block(rna, train_cells, eligible_cells, strategy="median", do_scale=True)
        X_train_parts.append(Xr_tr)
        X_all_parts.append(Xr_all)
        feature_names.extend([f"rna::{c}" for c in rna_cols])

    if "cnv" in feat_keys:
        Xc_tr, Xc_all, cnv_cols = build_nonpca_original_block(cnv, train_cells, eligible_cells, strategy="median", do_scale=True)
        X_train_parts.append(Xc_tr)
        X_all_parts.append(Xc_all)
        feature_names.extend([f"cnv::{c}" for c in cnv_cols])

    if "mut" in feat_keys:
        Xm_tr, Xm_all, mut_cols = build_nonpca_original_block(mut, train_cells, eligible_cells, strategy="most_frequent", do_scale=False)
        X_train_parts.append(Xm_tr)
        X_all_parts.append(Xm_all)
        feature_names.extend([f"mut::{c}" for c in mut_cols])

    if "prot" in feat_keys:
        Xp_tr, Xp_all, prot_cols = build_notmiwae_original_prot_block(
            prot_df=prot_core.loc[eligible_cells],
            train_cells=train_cells,
            all_cells=eligible_cells,
            missing_model=interp_missing_model,
            aux_train=aux_train,
            aux_all=aux_full,
        )
        X_train_parts.append(Xp_tr)
        X_all_parts.append(Xp_all)
        feature_names.extend([f"prot::{c}" for c in prot_cols])

    X_train_full = np.concatenate(X_train_parts, axis=1) if X_train_parts else np.zeros((len(train_cells), 0), dtype=np.float32)
    X_all_full = np.concatenate(X_all_parts, axis=1) if X_all_parts else np.zeros((len(eligible_cells), 0), dtype=np.float32)
    X_test_full = X_all_full[[cell_to_row[c] for c in test_cells]]

    pairs = drug_to_pairs[INTERP_DRUG].copy()
    pairs = pairs[pairs["depmap_id"].isin(eligible_cells)][["depmap_id", "y"]].groupby("depmap_id", as_index=False)["y"].mean()
    y_by_cell = dict(zip(pairs["depmap_id"].astype(str), pairs["y"].astype(float)))
    keep_train = [c for c in train_cells if c in y_by_cell]
    keep_test = [c for c in test_cells if c in y_by_cell]

    train_rows = [cell_to_row[c] for c in keep_train]
    test_rows = [cell_to_row[c] for c in keep_test]
    Xtr = X_all_full[train_rows]
    Xte = X_all_full[test_rows]
    ytr = np.array([y_by_cell[c] for c in keep_train], dtype=float)
    yte = np.array([y_by_cell[c] for c in keep_test], dtype=float)

    interp_model = make_model(INTERP_MODEL, INTERP_SEED)
    interp_model.fit(Xtr, ytr)
    pred = interp_model.predict(Xte)
    print("\nINTERPRETABILITY REFIT")
    print("  n_train:", len(ytr))
    print("  n_test :", len(yte))
    print("  Spearman:", spearman_corr(yte, pred))
    print("  R2      :", r2_score(yte, pred))

    interp_dir = OUT_REPORTS / "interpretability"
    interp_dir.mkdir(parents=True, exist_ok=True)

    # SHAP branch
    shap_feature_rank = None
    try:
        if INTERP_MODEL == "extratrees":
            explainer = shap.TreeExplainer(interp_model)
            shap_values = explainer.shap_values(Xte)
        else:
            bg = shap.sample(pd.DataFrame(Xtr, columns=feature_names), min(200, len(Xtr)), random_state=INTERP_SEED)
            explainer = shap.Explainer(interp_model.predict, bg)
            shap_values = explainer(pd.DataFrame(Xte, columns=feature_names)).values

        shap_abs = np.abs(np.asarray(shap_values)).mean(axis=0)
        shap_feature_rank = pd.DataFrame({
            "feature": feature_names,
            "mean_abs_shap": shap_abs,
        }).sort_values("mean_abs_shap", ascending=False)
        shap_feature_rank.to_csv(interp_dir / "shap_feature_rank.csv", index=False)
        print(f"[shap] wrote {interp_dir / 'shap_feature_rank.csv'}")
    except Exception as e:
        print(f"[shap] unavailable: {e}")

    # Integrated Gradients branch for an optional lightweight neural surrogate
    try:
        class SmallRegressor(nn.Module):
            def __init__(self, in_dim: int):
                super().__init__()
                self.net = nn.Sequential(
                    nn.Linear(in_dim, 256),
                    nn.ReLU(),
                    nn.Dropout(0.1),
                    nn.Linear(256, 64),
                    nn.ReLU(),
                    nn.Linear(64, 1),
                )

            def forward(self, x):
                return self.net(x).squeeze(-1)

        ig_model = SmallRegressor(Xtr.shape[1]).to(TORCH_DEVICE)
        opt = torch.optim.Adam(ig_model.parameters(), lr=1e-3)
        loss_fn = nn.MSELoss()
        xb = torch.from_numpy(Xtr.astype(np.float32)).to(TORCH_DEVICE)
        yb = torch.from_numpy(ytr.astype(np.float32)).to(TORCH_DEVICE)
        for _ in range(50):
            ig_model.train()
            pred_tr = ig_model(xb)
            loss = loss_fn(pred_tr, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()

        ig_model.eval()
        ig = IntegratedGradients(ig_model)
        x_test_t = torch.from_numpy(Xte.astype(np.float32)).to(TORCH_DEVICE)
        baseline = torch.zeros((1, Xte.shape[1]), dtype=torch.float32, device=TORCH_DEVICE)
        ig_attr = ig.attribute(x_test_t, baselines=baseline, n_steps=50)
        ig_abs = ig_attr.detach().cpu().numpy()
        ig_rank = pd.DataFrame({
            "feature": feature_names,
            "mean_abs_ig": np.abs(ig_abs).mean(axis=0),
        }).sort_values("mean_abs_ig", ascending=False)
        ig_rank.to_csv(interp_dir / "ig_feature_rank.csv", index=False)
        print(f"[ig] wrote {interp_dir / 'ig_feature_rank.csv'}")
    except Exception as e:
        print(f"[ig] unavailable: {e}")

    # Attention hook
    try:
        if hasattr(interp_model, "get_attention_weights"):
            attn = interp_model.get_attention_weights()
            pd.DataFrame(attn).to_csv(interp_dir / "attention_weights.csv", index=False)
            print(f"[attention] wrote {interp_dir / 'attention_weights.csv'}")
        else:
            print("[attention] skipped: current downstream refit is not an attention model.")
    except Exception as e:
        print(f"[attention] unavailable: {e}")

    # GNNExplainer hook
    try:
        if hasattr(interp_model, "edge_index") and hasattr(interp_model, "x_graph"):
            gnn_explainer = Explainer(
                model=interp_model,
                algorithm=GNNExplainer(epochs=200),
                explanation_type="model",
                node_mask_type="attributes",
                edge_mask_type="object",
                model_config=dict(mode="regression", task_level="graph", return_type="raw"),
            )
            explanation = gnn_explainer(interp_model.x_graph, interp_model.edge_index)
            if hasattr(explanation, "node_mask"):
                np.save(interp_dir / "gnn_node_mask.npy", explanation.node_mask.detach().cpu().numpy())
            if hasattr(explanation, "edge_mask"):
                np.save(interp_dir / "gnn_edge_mask.npy", explanation.edge_mask.detach().cpu().numpy())
            print(f"[gnnexplainer] wrote graph explanation artefacts to {interp_dir}")
        else:
            print("[gnnexplainer] skipped: current downstream refit is not a graph model.")
    except Exception as e:
        print(f"[gnnexplainer] unavailable: {e}")

    top_feature_names: List[str] = []
    if shap_feature_rank is not None and not shap_feature_rank.empty:
        top_feature_names = shap_feature_rank["feature"].head(TOP_N_FEATURES).tolist()
    try_run_pathway_enrichment(top_feature_names, interp_dir / "pathway_enrichment")



Merged not-MIWAE bake-off: artifacts/reports/notebook 7_notmiwae_prot/notmiwae_bakeoff_merged_3seeds.csv  shape=(2120400, 17)

not-MIWAE bake-off summary:


,config_rank,arm,model,feature_set,uses_prot,imputer_strategy,n_seeds,n_drugs,mean_spearman,median_spearman,std_spearman,mean_r2,baseline_mean_spearman,baseline_strategy,delta_vs_baseline
0,1,prot_combined_union,elasticnet,rna,False,no_prot_reference,3,100,0.209114,0.222888,0.115613,-0.061090,0.209114,no_prot_reference,0.000000
1,2,prot_combined_union,elasticnet,cnv,False,no_prot_reference,3,100,0.102522,0.093148,0.090596,-0.208090,0.102522,no_prot_reference,0.000000
2,3,prot_combined_union,elasticnet,mut,False,no_prot_reference,3,100,0.024459,0.021459,0.060367,-0.064916,0.024459,no_prot_reference,0.000000
3,4,prot_combined_union,elasticnet,prot,True,aux_rna_cnv_mut::notmiwae_linear,3,100,0.137279,0.151516,0.095574,-0.088369,NaN,NaN,NaN
4,4,prot_combined_union,elasticnet,prot,True,aux_rna_cnv_mut::notmiwae_linear+indicators,3,100,0.134120,0.143877,0.092848,-0.090442,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
739,180,prot_rppa_ccle,extratrees,rna+cnv+mut+prot,True,aux_rna_cnv_mut::notmiwae_linear+indicators,3,100,0.165976,0.169039,0.115183,-0.052743,0.179489,reference_without_prot,-0.013514
740,180,prot_rppa_ccle,extratrees,rna+cnv+mut+prot,True,notmiwae_linear,3,100,0.164228,0.170669,0.117898,-0.053530,0.179489,reference_without_prot,-0.015261
741,180,prot_rppa_ccle,extratrees,rna+cnv+mut+prot,True,notmiwae_linear+indicators,3,100,0.164228,0.170669,0.117898,-0.053530,0.179489,reference_without_prot,-0.015261
742,180,prot_rppa_ccle,extratrees,rna+cnv+mut+prot,True,notmiwae_selfmask,3,100,0.164228,0.170669,0.117898,-0.053530,0.179489,reference_without_prot,-0.015261



Per-seed not-MIWAE summary:


,seed,config_rank,arm,model,feature_set,uses_prot,imputer_strategy,n_drugs,mean_spearman,median_spearman,std_spearman,mean_r2
0,19537,1,prot_combined_union,elasticnet,rna,False,no_prot_reference,100,0.209441,0.222460,0.114188,-0.066557
744,1584678,1,prot_combined_union,elasticnet,rna,False,no_prot_reference,100,0.209097,0.223085,0.117295,-0.059761
1488,17052356,1,prot_combined_union,elasticnet,rna,False,no_prot_reference,100,0.208803,0.222083,0.115334,-0.056951
1,19537,2,prot_combined_union,elasticnet,cnv,False,no_prot_reference,100,0.102918,0.100161,0.090949,-0.207287
745,1584678,2,prot_combined_union,elasticnet,cnv,False,no_prot_reference,100,0.103162,0.088737,0.090542,-0.210338
...,...,...,...,...,...,...,...,...,...,...,...,...
2226,17052356,180,prot_rppa_ccle,extratrees,rna+cnv+mut+prot,True,aux_rna_cnv_mut::notmiwae_linear+indicators,100,0.164550,0.169398,0.114496,-0.052373
2227,17052356,180,prot_rppa_ccle,extratrees,rna+cnv+mut+prot,True,notmiwae_linear,100,0.163605,0.170669,0.119879,-0.053299
2228,17052356,180,prot_rppa_ccle,extratrees,rna+cnv+mut+prot,True,notmiwae_linear+indicators,100,0.163605,0.170669,0.119879,-0.053299
2229,17052356,180,prot_rppa_ccle,extratrees,rna+cnv+mut+prot,True,notmiwae_selfmask,100,0.163605,0.170669,0.119879,-0.053299


NOT-MIWAE LOCK DECISION

rank1__prot_combined_union__elasticnet__rna:
  Chosen   : not_applicable

rank2__prot_combined_union__elasticnet__cnv:
  Chosen   : not_applicable

rank3__prot_combined_union__elasticnet__mut:
  Chosen   : not_applicable

rank4__prot_combined_union__elasticnet__prot:
  Chosen   : aux_rna_cnv_mut::notmiwae_linear+indicators
  Spearman : 0.1341 ± 0.0928

rank5__prot_combined_union__elasticnet__rna+cnv:
  Chosen   : not_applicable

rank6__prot_combined_union__elasticnet__rna+mut:
  Chosen   : not_applicable

rank7__prot_combined_union__elasticnet__rna+prot:
  Chosen   : aux_rna_cnv_mut::notmiwae_linear+indicators
  Spearman : 0.1663 ± 0.1006
  Baseline : 0.2091  delta=-0.0428

rank8__prot_combined_union__elasticnet__cnv+mut:
  Chosen   : not_applicable

rank9__prot_combined_union__elasticnet__cnv+prot:
  Chosen   : aux_rna_cnv_mut::notmiwae_linear+indicators
  Spearman : 0.1225 ± 0.0878
  Baseline : 0.1025  delta=+0.0200

rank10__prot_combined_union__elasticnet__m